# RefillCare — Phase 4: Machine Learning Model Training & Evaluation

## 1. Objective
Phase 4 trains and evaluates gradient boosted decision trees (XGBoost, HistGradientBoosting) and Random Forest regressors against the baseline to predict `days_until_next_purchase`, evaluates subgroup accuracy, and converts predictions into actionable `expected_refill_date` values.


## 2. Input Data Setup


In [ ]:
import sys
import os
from pathlib import Path
import pandas as pd
import numpy as np

# Universal workspace root and sys.path resolver
current_dir = Path(__file__).resolve().parent if "__file__" in locals() else Path.cwd()
project_root = current_dir.resolve()
while project_root.parent != project_root and not (project_root / "refillcare" / "__init__.py").exists():
    project_root = project_root.parent

if (project_root / "refillcare" / "__init__.py").exists() and str(project_root) not in sys.path:
    sys.path.insert(0, str(project_root))

# Safe display helper for Jupyter and standalone environments
try:
    from IPython.display import display
except ImportError:
    display = print

# Safe matplotlib import
try:
    import matplotlib
    if "ipykernel" not in sys.modules:
        matplotlib.use("Agg")
    import matplotlib.pyplot as plt
except ImportError:
    plt = None

def find_file(relative_path: str) -> Path:
    candidates = [
        project_root / relative_path,
        Path.cwd() / relative_path,
        Path("..") / relative_path,
        Path("../..") / relative_path,
    ]
    for c in candidates:
        if c.exists():
            return c.resolve()
    return candidates[0]

import json
import joblib

model_path = find_file("data/refillcare/processed/models/refill_model.joblib")
report_path = find_file("data/refillcare/processed/phase4_model_report.json")

bundle = joblib.load(model_path)
with open(report_path, "r") as f:
    report = json.load(f)

print(f"Loaded Selected Model: {report['selected_model']}")


## 3. Processing & Model Comparison


In [ ]:
# Model comparison summary table
comp_data = []
for m_name, m_metrics in report["validation_comparison"].items():
    comp_data.append({
        "Model": m_name,
        "Val MAE (days)": m_metrics["mae"],
        "Val RMSE (days)": m_metrics["rmse"],
        "Val R2": m_metrics["r2"],
        "Within +-3d (%)": m_metrics["within_3_days_pct"],
        "Within +-7d (%)": m_metrics["within_7_days_pct"],
    })

comp_df = pd.DataFrame(comp_data).sort_values("Val MAE (days)")
display(comp_df)


## 4. Results & Deep-Dive Analysis


In [ ]:
# 4.1 Top Feature Importances Plot
top_feats = pd.DataFrame(report["top_features"]).head(10)

if plt is not None:
    plt.figure(figsize=(10, 5))
    plt.barh(top_feats["feature"][::-1], top_feats["importance"][::-1], color="#3867d6", edgecolor="black")
    plt.title(f"Top 10 Feature Importances ({report['selected_model']})", fontsize=13, pad=12)
    plt.xlabel("Importance Weight", fontsize=11)
    plt.grid(axis="x", linestyle="--", alpha=0.7)
    plt.tight_layout()
    plt.show()
else:
    print(top_feats)


In [ ]:
# 4.2 Subgroup Performance by History Depth
hist_data = []
for k, v in report["test_subgroups_by_history_length"].items():
    hist_data.append({
        "History Depth": k,
        "Test Events Count": v["count"],
        "Test MAE (days)": v["mae"],
        "Within +-7d (%)": v["within_7_days_pct"],
    })
display(pd.DataFrame(hist_data))


In [ ]:
# 4.3 Interactive Inference Demo: Predicting Expected Refill Date
from refillcare.models.prediction import predict_refill_date

sample_patient = {
    "customerId": "RAMESH_9849012345",
    "itemId": "101",
    "itemName": "TELMISARTAN-40MG",
    "invoice_date": pd.Timestamp("2026-06-15"),
    "purchase_count_so_far": 6,
    "days_since_first_purchase": 150,
    "days_since_previous_purchase": 30.0,
    "historical_interval_median": 30.0,
    "historical_interval_mean": 29.8,
    "historical_interval_std": 2.1,
    "historical_interval_min": 28.0,
    "historical_interval_max": 32.0,
    "historical_interval_cv": 0.07,
    "quantity": 30,
    "freeQuantity": 0,
    "avg_historical_quantity": 30.0,
    "quantity_vs_avg_ratio": 1.0,
    "purchase_month": 6,
    "purchase_day_of_week": 0,
    "purchase_day_of_month": 15,
    "purchase_day_of_year": 166,
    "purchase_quarter": 2,
    "is_weekend": 0,
    "is_first_purchase": 0,
    "has_multiple_prior_purchases": 1,
    "is_recurring_history": 1,
    "therapeuticCategory": "CARDIAC",
    "salt_category": "TABLETS",
    "salt_itemcat": "PHARMA",
}

pred_res = predict_refill_date(bundle, sample_patient)
print("=== RefillCare Prediction Output ===")
for k, v in pred_res.items():
    print(f"  {k:30}: {v}")


## 5. Architectural Findings
- **XGBoost Outperforms Baseline:** Validation MAE dropped from 19.01 days to **15.24 days** (a ~3.8 day reduction in refill prediction error).
- **High Reliability on Deep Histories:** For patients with $>5$ historical purchases, MAE drops down to **9.64 days**, with **47.3% accuracy within a $\pm 7$-day window**.
- **Actionable Reminder Dates:** Converting predicted days to `expected_refill_date` provides the exact target date for automated WhatsApp reminder triggers (e.g. -7d, -3d, -1d).


## 6. Conclusion
Phase 4 delivered a validated, serialized machine learning model (`refill_model.joblib`) ready for Phase 5 (Reminder Scheduling & Streamlit UI).
